# Eval: Intermediate Plots (Detailed)

This notebook unrolls the `eval.plot_intermediate.plot_intermediate` workflow into explicit Python steps.

What you get:
- how sampling steps are selected
- how an `inter_state` dataset is inspected
- how trajectory plots are produced and displayed inline


In [ ]:
from pathlib import Path
import json
import xarray as xr
import matplotlib.pyplot as plt
from IPython.display import Image, display

from eval.plot_intermediate.plot_intermediate import (
    select_sampling_steps,
    resolve_capture_steps,
    _parse_steps_csv,
    plot_intermediate_trajectory,
)

plt.rcParams['figure.figsize'] = (10, 4)


In [ ]:
# Step 1: define sampling config and inspect capture-step logic
extra_args_json = '{"num_steps":40,"sigma_max":1000.0,"sigma_min":0.03,"rho":7.0,"sampler":"heun"}'
extra_args = json.loads(extra_args_json)

total_steps = int(extra_args.get('num_steps', 40))
auto_steps = resolve_capture_steps(total_steps=total_steps, explicit_steps=None, capture_max_steps=8)
manual_steps = _parse_steps_csv('0,3,7,15,23,31,39')
manual_steps = resolve_capture_steps(total_steps=total_steps, explicit_steps=manual_steps)

print('total_steps =', total_steps)
print('auto capture steps =', auto_steps)
print('manual capture steps =', manual_steps)
print('quick panel positions for preview =', select_sampling_steps(total_steps, max_panels=6))


In [ ]:
# Step 2: point to a dataset containing `inter_state`
# Option A: existing dataset (recommended for notebook usage)
predictions_nc = Path('/path/to/predictions_with_intermediate.nc')

# Option B: generate with CLI first, then re-open here:
# python -m eval.plot_intermediate.plot_intermediate checkpoint \
#   --name-ckpt <RUN_ID_or_ckpt_path> \
#   --weather-state 2t --out /tmp/intermediate_2t.png \
#   --save-intermediate-nc /tmp/intermediate_states.nc

if not predictions_nc.exists():
    print(f'Missing file: {predictions_nc}')
    print('Set `predictions_nc` to a real file path before running the next cells.')


In [ ]:
# Step 3: inspect dataset structure
if predictions_nc.exists():
    ds = xr.open_dataset(predictions_nc)
    print(ds)
    print('
Dims:', ds.dims)
    print('Vars:', list(ds.data_vars))
    print('Weather states:', ds.weather_state.values.tolist())
    if 'sampling_step' in ds.coords:
        print('Sampling steps:', ds.sampling_step.values.tolist())


In [ ]:
# Step 4: pick one sample/member/state and plot trajectory (inline)
out_png = Path('/tmp/intermediate_walkthrough.png')
weather_state = '2t'
sample = 0
member = 0
region = 'default'

if predictions_nc.exists():
    out = plot_intermediate_trajectory(
        ds=ds,
        weather_state=weather_state,
        sample=sample,
        member=member,
        out_path=str(out_png),
        region=region,
        max_panels=8,
        sampling_steps=None,
    )
    print('Saved:', out)
    display(Image(filename=str(out)))


In [ ]:
# Step 5: inspect one variable through sampling steps directly in Python
if predictions_nc.exists() and 'inter_state' in ds:
    sel = ds.sel(sample=0, ensemble_member=0, weather_state='2t')
    means = sel['inter_state'].mean(dim='grid_point_hres').values
    steps = sel['sampling_step'].values

    plt.figure()
    plt.plot(steps, means, marker='o')
    plt.xlabel('sampling_step')
    plt.ylabel('mean inter_state (2t)')
    plt.title('Trajectory summary (inline)')
    plt.grid(True)
    plt.show()
